# HTTPS用CAの更新

新しいCAとホスト証明書へ切り替える。D11aで準備した証明書を使用するクラスタが対象。新旧CAを信頼する期間を設け、ホストごとの切り替えを確認してから旧CAの信頼を外す。

HTTPSサービスの再起動を伴う。新規ジョブの投入を止め、実行中のジョブが完了していること、HDFSのHAと複製が健全であることを確認して作業する。


## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*


# 対象クラスタの設定


In [ ]:
target_group = 'hadoop_all_testcluster'
target_group


In [ ]:
import os
import yaml

with open('group_vars/' + target_group) as stream:
    configuration = yaml.safe_load(stream)


inventory上の接続先と、証明書に記載するホスト名を指定する。接続先のホスト名を確認する。


In [ ]:
!ansible {target_group} -a 'hostname -f'


In [ ]:
certificate_hosts = {
    'xxx.xxx.xxx.64': 'test01',
    'xxx.xxx.xxx.65': 'test02',
}
certificate_hosts


HTTPSで確認するサービスのホスト名とポートを指定する。リスニングポートと現在の設定を確認する。


In [ ]:
!ansible {target_group} --become -a 'ss -lntp'


In [ ]:
https_endpoints = [
    ('test01', 8481), ('test02', 8481),
    ('test01', 50470), ('test02', 50470),
    ('test01', 50475), ('test02', 50475),
    ('test01', 8088), ('test02', 8088),
    ('test01', 8044), ('test02', 8044),
    ('test01', 19890), ('test01', 8190),
]
https_endpoints


# 更新前の確認

利用者として認証し、実行中のYARNアプリケーションがないことを確認する。


In [ ]:
!ansible hadoop_client -l {target_group} -m shell -a 'kinit -kt "$HOME/.hadoop.keytab" "$(id -un)"'


In [ ]:
!ansible hadoop_client -l {target_group} -a 'yarn application -list -appStates NEW,NEW_SAVING,SUBMITTED,ACCEPTED,RUNNING'


NameNodeとJournalNode上でhdfsユーザーとして認証し、HAの状態とDataNode・複製の状態を確認する。


In [ ]:
!ansible hadoop_namenode -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'
!ansible hadoop_journalnode -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'


In [ ]:
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState'
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs dfsadmin -report'


# 新しいCAの準備

更新対象CAの公開証明書と、新しいCAを保管するディレクトリを指定する。新しいCAの鍵は新規に作成する。


In [ ]:
old_ca_certificate_file = os.path.expanduser('~/.local/share/hadoop-pki/initial/ca.crt')
pki_dir = os.path.expanduser('~/.local/share/hadoop-pki/replacement')
(old_ca_certificate_file, pki_dir)


In [ ]:
!openssl x509 -in "{old_ca_certificate_file}" -noout -text -fingerprint -sha256
!openssl x509 -in "{old_ca_certificate_file}" -noout -checkend 0


In [ ]:
ca_subject = '/CN=Hadoop Root CA replacement'
ca_days = 3650
certificate_days = 365
(ca_subject, ca_days, certificate_days)


In [ ]:
ca_key_file = os.path.join(pki_dir, 'ca.key')
ca_certificate_file = os.path.join(pki_dir, 'ca.crt')
truststore_file = os.path.join(pki_dir, 'truststore.jks')
(ca_key_file, ca_certificate_file, truststore_file)


In [ ]:
!openssl version
!keytool -help


CA秘密鍵のパスワードを入力する。ホストのkeystoreとtruststoreには、group_varsに指定したパスワードを使用する。


In [ ]:
import getpass

os.environ['HADOOP_CA_PASSWORD'] = getpass.getpass('CA key password: ')
os.environ['HADOOP_KEYSTORE_PASSWORD'] = configuration['https_keystore_pass']
os.environ['HADOOP_PRIVATE_KEY_PASSWORD'] = configuration['https_privkey_pass']
os.environ['HADOOP_TRUSTSTORE_PASSWORD'] = configuration['https_truststore_pass']


In [ ]:
!install -d -m 0700 "{pki_dir}"
!test ! -e "{ca_key_file}"
!test ! -e "{ca_certificate_file}"


ルートCAの鍵と自己署名証明書を作成する。


In [ ]:
!umask 077; openssl req -new -x509 -newkey rsa:3072 -sha256 -days {ca_days} \
    -subj "{ca_subject}" \
    -addext "basicConstraints=critical,CA:TRUE,pathlen:0" \
    -addext "keyUsage=critical,keyCertSign,cRLSign" \
    -passout env:HADOOP_CA_PASSWORD \
    -keyout "{ca_key_file}" \
    -out "{ca_certificate_file}"


CAのSubject、有効期限、CAとしての制約とフィンガープリントを確認する。


In [ ]:
!openssl x509 -in "{ca_certificate_file}" -noout -text -fingerprint -sha256


# Playbookの準備


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
work_dir


In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1


In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir


# 新旧CAを信頼する設定

新しい証明書を配る前に、すべてのノードへ新旧CAを含むtruststoreを配布する。

クラスタ外のクライアントにも、新旧CAを信頼する設定を反映して接続を確認する。


In [ ]:
!keytool -importcert -noprompt -alias root-ca -file "{ca_certificate_file}" \
    -keystore "{truststore_file}" -storetype JKS \
    -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!keytool -importcert -noprompt -alias old-root-ca -file "{old_ca_certificate_file}" \
    -keystore "{truststore_file}" \
    -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!keytool -list -v -keystore "{truststore_file}" -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
ca_bundle_file = os.path.join(pki_dir, 'ca-certificates.pem')
ca_bundle_file


In [ ]:
!cat "{old_ca_certificate_file}" "{ca_certificate_file}" > "{ca_bundle_file}"


In [ ]:
https_keystore_path = configuration['https_keystore_path']
https_truststore_path = configuration['https_truststore_path']
ca_bundle_path = '/etc/hadoop/conf/ca-certificates.pem'
(https_keystore_path, https_truststore_path, ca_bundle_path)


In [ ]:
next_ca_path = '/etc/hadoop/conf/ca-next.crt'
next_ca_path


In [ ]:
!ansible {target_group} --become -m copy -a 'src={truststore_file} dest={https_truststore_path} owner=root group=root mode=0444'
!ansible {target_group} --become -m copy -a 'src={ca_bundle_file} dest={ca_bundle_path} owner=root group=root mode=0444'
!ansible {target_group} --become -m copy -a 'src={ca_certificate_file} dest={next_ca_path} owner=root group=root mode=0444'


HTTPSサービスを1台ずつ再起動してtruststoreを読み直す。NameNodeはJournalNodeからHTTPSでログを取得するため、JournalNodeを先に再起動する。JournalNodeは1台ごとにログをロールして書き込みへ復帰させてから、次のサーバーへ進む。


In [ ]:
!ansible-playbook {playbook_dir}/restart_https_services.yml -l {target_group}


新旧CAを含む設定で、現在の証明書とのTLS接続が検証できることを確認する。


In [ ]:
for hostname, port in https_endpoints:
    !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {ca_bundle_path} -connect {hostname}:{port} < /dev/null'


# ホスト証明書の発行

ホストごとに新しい鍵とCSRを作成する。ホスト名はSubject Alternative Nameに記載する。


In [ ]:
for address, hostname in certificate_hosts.items():
    !umask 077; keytool -genkeypair -alias server -keyalg RSA -keysize 3072 \
        -dname "CN={hostname}" -ext "SAN=dns:{hostname}" -storetype JKS \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD \
        -keypass:env HADOOP_PRIVATE_KEY_PASSWORD


In [ ]:
for address, hostname in certificate_hosts.items():
    !keytool -certreq -alias server -ext "SAN=dns:{hostname}" \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD \
        -keypass:env HADOOP_PRIVATE_KEY_PASSWORD -file "{pki_dir}/{hostname}.csr"


In [ ]:
for address, hostname in certificate_hosts.items():
    !openssl req -in "{pki_dir}/{hostname}.csr" -noout -text -verify


発行する証明書の用途とホスト名を指定する。


In [ ]:
for address, hostname in certificate_hosts.items():
    extension_file = os.path.join(pki_dir, hostname + '.ext')
    with open(extension_file, 'w') as stream:
        stream.write('basicConstraints=critical,CA:FALSE\nkeyUsage=critical,digitalSignature,keyEncipherment\nextendedKeyUsage=serverAuth\nsubjectAltName=DNS:' + hostname + '\n')
    !cat "{extension_file}"


In [ ]:
for address, hostname in certificate_hosts.items():
    !openssl x509 -req -in "{pki_dir}/{hostname}.csr" -CA "{ca_certificate_file}" -CAkey "{ca_key_file}" \
        -passin env:HADOOP_CA_PASSWORD -CAcreateserial -days {certificate_days} -sha256 \
        -extfile "{pki_dir}/{hostname}.ext" \
        -out "{pki_dir}/{hostname}.crt"


発行元、有効期限、ホスト名を確認し、CA証明書で検証する。


In [ ]:
for address, hostname in certificate_hosts.items():
    !openssl x509 -in "{pki_dir}/{hostname}.crt" -noout -text -fingerprint -sha256
    !openssl verify -CAfile "{ca_certificate_file}" -verify_hostname "{hostname}" "{pki_dir}/{hostname}.crt"


発行した証明書を、鍵を作成したkeystoreへ取り込む。


In [ ]:
for address, hostname in certificate_hosts.items():
    !keytool -importcert -noprompt -alias root-ca -file "{ca_certificate_file}" \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD
    !keytool -importcert -noprompt -alias server -file "{pki_dir}/{hostname}.crt" \
        -keystore "{pki_dir}/{hostname}-keystore.jks" \
        -storepass:env HADOOP_KEYSTORE_PASSWORD \
        -keypass:env HADOOP_PRIVATE_KEY_PASSWORD


In [ ]:
for address, hostname in certificate_hosts.items():
    !keytool -list -v -keystore "{pki_dir}/{hostname}-keystore.jks" -storepass:env HADOOP_KEYSTORE_PASSWORD


# ホスト証明書の切り替え

1台ずつ新しいkeystoreへ交換してHTTPSサービスを再起動する。そのホストの接続を新CAだけで検証できることを確認してから次へ進む。


In [ ]:
for address, hostname in certificate_hosts.items():
    !ansible {address} --become -m copy -a 'src={pki_dir}/{hostname}-keystore.jks dest={https_keystore_path} owner=root group=hadoop mode=0440'
    !ansible-playbook {playbook_dir}/restart_https_services.yml -l {address}
    for endpoint_hostname, port in https_endpoints:
        if endpoint_hostname == hostname:
            !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {next_ca_path} -connect {hostname}:{port} < /dev/null'


# 旧CAの信頼を外す

すべての接続先が新CAだけで検証できることを確認する。旧CAで発行した証明書を使う接続先が残っている場合は、切り替えを済ませてから進む。


In [ ]:
for hostname, port in https_endpoints:
    !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {next_ca_path} -connect {hostname}:{port} < /dev/null'


In [ ]:
!keytool -delete -alias old-root-ca -keystore "{truststore_file}" -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!keytool -list -v -keystore "{truststore_file}" -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!ansible {target_group} --become -m copy -a 'src={truststore_file} dest={https_truststore_path} owner=root group=root mode=0444'
!ansible {target_group} --become -m copy -a 'src={ca_certificate_file} dest={ca_bundle_path} owner=root group=root mode=0444'


In [ ]:
!ansible-playbook {playbook_dir}/restart_https_services.yml -l {target_group}


# 更新後の確認

新CAだけを含む設定でTLS接続を確認する。


In [ ]:
for hostname, port in https_endpoints:
    !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {ca_bundle_path} -connect {hostname}:{port} < /dev/null'


In [ ]:
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState'
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs dfsadmin -report'
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs fsck /'


# 後始末

配置済みのホスト秘密鍵と、検証に使用した公開証明書の一時ファイルを削除する。新CAの鍵と証明書は今後の発行に使用するため保管する。


In [ ]:
for address, hostname in certificate_hosts.items():
    !rm "{pki_dir}/{hostname}-keystore.jks"


In [ ]:
!ansible {target_group} --become -m file -a 'path={next_ca_path} state=absent'
!rm -fr {work_dir}


In [ ]:
del os.environ['HADOOP_CA_PASSWORD']
del os.environ['HADOOP_KEYSTORE_PASSWORD']
del os.environ['HADOOP_PRIVATE_KEY_PASSWORD']
del os.environ['HADOOP_TRUSTSTORE_PASSWORD']
